In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import torch
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src/kldm").exists() and (candidate / "src/kldm/kldm.py").exists():
            return candidate
    raise RuntimeError("Could not find repo root containing src/kldm")


REPO_ROOT = find_repo_root(Path.cwd())
SRC_ROOT = REPO_ROOT / "src"

# Avoid the common notebook import-shadowing issue when running from src/kldm
if Path.cwd().resolve() == (REPO_ROOT / "src/kldm").resolve():
    os.chdir(REPO_ROOT)

bad_kldm = sys.modules.get("kldm")
if bad_kldm is not None and not hasattr(bad_kldm, "__path__"):
    del sys.modules["kldm"]

sys.path[:] = [
    p for p in sys.path
    if Path(p).resolve() != (REPO_ROOT / "src/kldm").resolve()
]
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

try:
    from kldm.data import CSPTask, resolve_data_root
    from kldm.diffusionModels.trivialized_diffusion import TrivialisedDiffusion
    from kldm.diffusionModels.lambda_t import precompute_lambda_time_grid_from_loader
    from kldm.kldm import ModelKLDM
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Notebook imports failed. Run this notebook in the project environment that has the KLDM dependencies installed."
    ) from exc


def show_source(rel_path: str, start: int | None = None, end: int | None = None, title: str | None = None):
    path = REPO_ROOT / rel_path
    text = path.read_text(encoding="utf-8").splitlines()
    start = 1 if start is None else start
    end = len(text) if end is None else end
    width = len(str(end))
    snippet = "\n".join(f"{i:>{width}} | {text[i-1]}" for i in range(start, end + 1))
    heading = title or rel_path
    display(Markdown(
        f"### {heading}\n\n"
        f"`{rel_path}:{start}-{end}`\n"
        f"```python\n{snippet}\n```"
    ))


def summarize_tensor(name: str, x: torch.Tensor):
    x = x.detach()
    print(
        f"{name}: shape={tuple(x.shape)} dtype={x.dtype} "
        f"mean={float(x.float().mean()):.6f} std={float(x.float().std()):.6f} "
        f"min={float(x.min()):.6f} max={float(x.max()):.6f}"
    )


def summarize_batch(batch):
    print(f"num_graphs={int(batch.num_graphs)} num_nodes={int(batch.pos.shape[0])}")
    for key in ["pos", "h", "l", "batch", "edge_node_index"]:
        if hasattr(batch, key):
            value = getattr(batch, key)
            if torch.is_tensor(value):
                print(f"{key}: shape={tuple(value.shape)} dtype={value.dtype}")
            else:
                print(f"{key}: {type(value)}")


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"REPO_ROOT={REPO_ROOT}")
print(f"device={device}")


MODELS_PROJECT_ROOT: /workspace/.venv/lib/python3.11/site-packages/mattergen
REPO_ROOT=/workspace
device=cpu


In [2]:
ROOT = resolve_data_root()
SPLIT = 'train'
BATCH_SIZE = 2
NUM_WORKERS = 0
DOWNLOAD = False
LAMBDA_NUM_BATCHES = 2
SAMPLING_STEPS = 8

print(f'ROOT={ROOT}')
print(f'SPLIT={SPLIT} BATCH_SIZE={BATCH_SIZE} NUM_WORKERS={NUM_WORKERS} DOWNLOAD={DOWNLOAD}')
print(f'LAMBDA_NUM_BATCHES={LAMBDA_NUM_BATCHES} SAMPLING_STEPS={SAMPLING_STEPS}')


ROOT=/workspace/data
SPLIT=train BATCH_SIZE=2 NUM_WORKERS=0 DOWNLOAD=False
LAMBDA_NUM_BATCHES=2 SAMPLING_STEPS=8


## 1. Data Handler And CSP Transforms

These cells show the exact source and then instantiate a real `CSPTask` dataloader.


In [3]:
show_source('src/kldm/data/csp.py', 1, 44, 'CSPTask')
show_source('src/kldm/data/transform.py', 34, 48, 'FullyConnectedGraph')
show_source('src/kldm/data/transform.py', 125, 210, 'ContinuousIntervalLattice')
show_source('src/kldm/data/transform.py', 359, 385, 'CopyProperty + TaskMetadata')


### CSPTask

`src/kldm/data/csp.py:1-44`
```python
 1 | from __future__ import annotations
 2 | 
 3 | from pathlib import Path
 4 | 
 5 | from torch.utils.data import DataLoader
 6 | 
 7 | from .dataset import MP20, resolve_data_root
 8 | from .transform import ContinuousIntervalLattice, CopyProperty, DEFAULT_ATOMIC_VOCAB, FullyConnectedGraph, OneHot, TaskMetadata
 9 | 
10 | TASK_CSP = 0
11 | 
12 | 
13 | class CSPTask:
14 |     """Return CSP-ready MatterGen batches."""
15 | 
16 |     def __init__(self, species_vocab: list[int] | None = None) -> None:
17 |         self.species_vocab = species_vocab or DEFAULT_ATOMIC_VOCAB
18 |         self.transforms = [
19 |             FullyConnectedGraph(),
20 |             ContinuousIntervalLattice(),
21 |             CopyProperty("atomic_numbers", "h"),
22 |             TaskMetadata(task_id=TASK_CSP, diffuse_h=False),
23 |         ]
24 | 
25 |     def fit_dataset(self, root: str | Path | None = None, split: str = "train", download: bool = False) -> MP20:
26 |         return MP20(root=resolve_data_root(root), split=split, transforms=self.transforms, download=download)
27 | 
28 |     def dataloader(
29 |         self,
30 |         root: str | Path | None = None,
31 |         split: str = "train",
32 |         batch_size: int = 32,
33 |         shuffle: bool = True,
34 |         num_workers: int = 2,
35 |         download: bool = False,
36 |     ) -> DataLoader:
37 |         dataset = self.fit_dataset(root=root, split=split, download=download)
38 |         return DataLoader(
39 |             dataset,
40 |             batch_size=batch_size,
41 |             shuffle=shuffle,
42 |             num_workers=num_workers,
43 |             collate_fn=dataset.collate_fn,
44 |         )
```

### FullyConnectedGraph

`src/kldm/data/transform.py:34-48`
```python
34 | @functional_transform("fully_connected_graph")
35 | class FullyConnectedGraph(Transform):
36 |     """Create a fully connected graph stored in `edge_node_index`."""
37 | 
38 |     def __init__(self, key: str = "edge_node_index", len_from: str = "pos") -> None:
39 |         self.key = key
40 |         self.len_from = len_from
41 | 
42 |     def __call__(self, sample: ChemGraph) -> ChemGraph:
43 |         # KLDM uses a dense all-to-all graph for message passing in these examples.
44 |         n = len(getattr(sample, self.len_from))
45 |         fc_graph = torch.ones(n, n, device=sample.pos.device) - torch.eye(n, device=sample.pos.device)
46 |         fc_edges, _ = dense_to_sparse(fc_graph)
47 |         return sample.replace(**{self.key: fc_edges})
48 | 
```

### ContinuousIntervalLattice

`src/kldm/data/transform.py:125-210`
```python
125 | @functional_transform("continuous_interval_lattice")
126 | class ContinuousIntervalLattice(Transform):
127 |     """
128 |     Lattice transform for KLDM.
129 | 
130 |     Forward:
131 |         cell -> [log(lengths), tan(angle - pi/2)] -> optional standardization
132 | 
133 |     Inverse:
134 |         standardized_l -> unconstrained_l -> physical lengths/angles
135 |     """
136 | 
137 |     def __init__(
138 |         self,
139 |         out_key: str = "l",
140 |         normalize_lengths_by_num_atoms: bool = False,
141 |         cache_file: str | Path | None = None,
142 |         quantile: float = 0.025,
143 |         standardize: bool = True,
144 |         standardize_by_num_atoms: bool = True,
145 |         eps: float = 1e-8,
146 |     ) -> None:
147 |         self.out_key = out_key
148 |         self.normalize_lengths_by_num_atoms = normalize_lengths_by_num_atoms
149 |         self.cache_file = Path(cache_file) if cache_file is not None else None
150 |         self.quantile = float(quantile)
151 |         self.standardize = bool(standardize)
152 |         self.standardize_by_num_atoms = bool(standardize_by_num_atoms)
153 |         self.eps = float(eps)
154 | 
155 |         # key -> (loc[6], scale[6])
156 |         self.loc_scale: dict[int | str, tuple[torch.Tensor, torch.Tensor]] = {}
157 | 
158 |         if self.cache_file is not None and self.cache_file.exists():
159 |             with self.cache_file.open("r", encoding="utf-8") as f:
160 |                 raw = json.load(f)
161 |             self.loc_scale = {
162 |                 self._decode_key(k): (
163 |                     torch.tensor(v["loc"], dtype=torch.get_default_dtype()),
164 |                     torch.tensor(v["scale"], dtype=torch.get_default_dtype()),
165 |                 )
166 |                 for k, v in raw.items()
167 |             }
168 | 
169 |     def _group_key(self, n_atoms: int) -> int | str:
170 |         return int(n_atoms) if self.standardize_by_num_atoms else "global"
171 | 
172 |     @staticmethod
173 |     def _decode_key(k: str) -> int | str:
174 |         return "global" if k == "global" else int(k)
175 | 
176 |     def _encode_key(self, k: int | str) -> str:
177 |         return str(k)
178 | 
179 |     def _to_unconstrained_lattice(self, cell_matrix: torch.Tensor, n_atoms: int) -> torch.Tensor:
180 |         lengths, angles_rad = _cell_lengths_and_angles(cell_matrix)
181 | 
182 |         if self.normalize_lengths_by_num_atoms:
183 |             lengths = lengths / (float(n_atoms) ** (1.0 / 3.0))
184 | 
185 |         log_lengths = torch.log(lengths.clamp_min(self.eps))
186 |         angle_features = torch.tan(angles_rad - torch.pi / 2)
187 | 
188 |         return torch.cat([log_lengths, angle_features], dim=0)
189 | 
190 |     def __call__(self, sample: ChemGraph) -> ChemGraph:
191 |         if not hasattr(sample, "cell"):
192 |             raise ValueError("ChemGraph must have a 'cell' attribute to use ContinuousIntervalLattice transform.")
193 | 
194 |         n_atoms = int(sample.num_atoms)
195 |         cell_matrix = sample.cell.squeeze(0)
196 |         l_unscaled = self._to_unconstrained_lattice(cell_matrix=cell_matrix, n_atoms=n_atoms)
197 | 
198 |         l = l_unscaled.clone()
199 |         if self.standardize:
200 |             key = self._group_key(n_atoms)
201 |             if key in self.loc_scale:
202 |                 loc, scale = self.loc_scale[key]
203 |                 l = (l - loc) / scale.clamp_min(self.eps)
204 | 
205 |         return sample.replace(
206 |             **{
207 |                 self.out_key: l.view(1, 6),
208 |                 "l_unscaled": l_unscaled.view(1, 6),
209 |             }
210 |         )
```

### CopyProperty + TaskMetadata

`src/kldm/data/transform.py:359-385`
```python
359 | class CopyProperty(Transform):
360 |     """Copy one attribute to another attribute name."""
361 | 
362 |     def __init__(self, in_key: str, out_key: str) -> None:
363 |         self.in_key = in_key
364 |         self.out_key = out_key
365 | 
366 |     def __call__(self, sample: ChemGraph) -> ChemGraph:
367 |         return sample.replace(**{self.out_key: getattr(sample, self.in_key)})
368 | 
369 | 
370 | class TaskMetadata(Transform):
371 |     """Attach KLDM task metadata to a ChemGraph."""
372 | 
373 |     def __init__(self, task_id: int, diffuse_h: bool, is_prior: bool = False) -> None:
374 |         self.task_id = int(task_id)
375 |         self.diffuse_h = bool(diffuse_h)
376 |         self.is_prior = bool(is_prior)
377 | 
378 |     def __call__(self, sample: ChemGraph) -> ChemGraph:
379 |         # These small flags are enough for KLDM to distinguish CSP vs DNG batches.
380 |         return sample.replace(
381 |             task_id=torch.tensor([self.task_id], dtype=torch.long),
382 |             diffuse_h=torch.tensor([self.diffuse_h], dtype=torch.bool),
383 |             is_prior=torch.tensor([self.is_prior], dtype=torch.bool),
384 |             num_atoms=torch.tensor([int(sample.num_atoms)], dtype=torch.long),
385 |         )
```

In [4]:
task = CSPTask()
loader = task.dataloader(
    root=ROOT,
    split=SPLIT,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    download=DOWNLOAD,
)

batch = next(iter(loader))
summarize_batch(batch)


num_graphs=2 num_nodes=25
pos: shape=(25, 3) dtype=torch.float32
h: shape=(25,) dtype=torch.int64
l: shape=(2, 6) dtype=torch.float32
batch: shape=(25,) dtype=torch.int64
edge_node_index: shape=(2, 288) dtype=torch.int64


## 2. TDM Setup

This is the live `TrivialisedDiffusion` class that the project is using right now.


In [5]:
show_source('src/kldm/diffusionModels/trivialized_diffusion.py', 28, 124, 'TrivialisedDiffusion setup + lambda hooks')


### TrivialisedDiffusion setup + lambda hooks

`src/kldm/diffusionModels/trivialized_diffusion.py:28-124`
```python
 28 | class TrivialisedDiffusion(nn.Module):
 29 |     """
 30 |     trivialised diffusion for positions + velocities.
 31 |     """
 32 |     def __init__(
 33 |             self,
 34 |             eps: float = 1e-5,
 35 |             n_lambdas: int = 256,
 36 |             lambda_num_batches: int = 16) -> None:
 37 |         super().__init__()
 38 |         self.eps = float(eps)
 39 |         self.time_scaling_T = 2
 40 |         self.n_lambdas = int(n_lambdas)
 41 |         self.lambda_num_batches = int(lambda_num_batches)
 42 |         self.register_buffer("_lambda_t01_grid", torch.linspace(1e-4, 1.0, self.n_lambdas))
 43 |         self.register_buffer("_lambda_v_table", torch.ones(self.n_lambdas))
 44 | 
 45 |     # -------------------------------------------------
 46 |     #  Wrapping function.
 47 |     # -------------------------------------------------
 48 | 
 49 |     @staticmethod
 50 |     def wrap_positions(x: torch.Tensor) -> torch.Tensor:
 51 |         """Wrap unit-cell fractional coordinates into [0, 1)."""
 52 |         return torch.remainder(x, 1.0)
 53 | 
 54 |     #displacements usually should be in [-0.5,0.5), see report.
 55 |     #
 56 |     #Måske ikke nødvendigt.
 57 |     @staticmethod
 58 |     def wrap_displacements(x: torch.Tensor) -> torch.Tensor:
 59 |         """Wrap signed periodic displacements into [-0.5, 0.5)."""
 60 |         return torch.remainder(x + 0.5, 1.0) - 0.5
 61 | 
 62 |     # -------------------------------------------------
 63 |     # Velocity sampling
 64 |     # -------------------------------------------------
 65 | 
 66 |     # v_t | v_0 ~ N(exp(-t) v_0, (1 - exp(-2t)) I)
 67 |     def gaussian_velocity_mean_coeff(self, t: torch.Tensor) -> torch.Tensor:
 68 |         """Mean coefficient of the Gaussian velocity forward kernel."""
 69 |         return torch.exp(-t)
 70 | 
 71 |     def gaussian_velocity_sigma(self, t: torch.Tensor) -> torch.Tensor:
 72 |         """Standard deviation of the Gaussian velocity forward kernel."""
 73 |         return torch.sqrt(torch.clamp(1.0 - torch.exp(-2.0 * t), min=self.eps))
 74 | 
 75 |     def wrapped_gaussian_mu_r_t_pre_wrap(self, t: torch.Tensor, v_t: torch.Tensor, v0: torch.Tensor) -> torch.Tensor:
 76 |         """Unwrapped wrapped-Gaussian mean in the internal unit-period chart."""
 77 |         coeff = (1.0 - torch.exp(-t)) / (1.0 + torch.exp(-t))  # Eq. (22)
 78 |         coeff = self._match_dims(coeff, v_t)
 79 |         return coeff * (v_t + v0)
 80 | 
 81 |     def wrapped_gaussian_mu_r_t(self, t: torch.Tensor, v_t: torch.Tensor, v0: torch.Tensor) -> torch.Tensor:
 82 |         """Backward-compatible alias for the pre-wrap mean helper."""
 83 |         return self.wrapped_gaussian_mu_r_t_pre_wrap(t=t, v_t=v_t, v0=v0)
 84 | 
 85 |     def wrapped_gaussian_sigma_r_t(self, t: torch.Tensor) -> torch.Tensor:
 86 |         return torch.sqrt(
 87 |             torch.clamp(2.0 * t + 8.0 / (1.0 + torch.exp(t)) - 4.0, min=self.eps)
 88 |         )  # Eq. (23)
 89 | 
 90 |     def lambda_v(self, t01: torch.Tensor) -> torch.Tensor:
 91 |         """
 92 |         Interpolate λ(t) for normalized time t01 in [0,1].
 93 |         """
 94 |         return interpolate_lambda_table(self._lambda_v_table, t01)
 95 | 
 96 |     # NO GRAD HERE!!!, THIS IS PURELY PRECOMPUTED!!!
 97 |     @torch.no_grad()
 98 |     def precompute_lambda_v_table_from_loader(
 99 |         self,
100 |         loader,
101 |         device: torch.device | None = None,
102 |         num_batches: int | None = None,
103 |         clamp_min: float = 0.2,
104 |         clamp_max: float = 5.0,
105 |         smooth: bool = True,
106 |     ) -> torch.Tensor:
107 |         """
108 |         Precompute λ(t) from the exact wrapped-normal target used in training.
109 | 
110 |         This matches the actual training loss:
111 |             target = score_target(...)
112 |         """
113 |         lambda_table = precompute_lambda_time_grid_from_loader(
114 |             diffusion=self,
115 |             loader=loader,
116 |             t01_grid=self._lambda_t01_grid,
117 |             num_batches=self.lambda_num_batches if num_batches is None else int(num_batches),
118 |             clamp_min=clamp_min,
119 |             clamp_max=clamp_max,
120 |             smooth=smooth,
121 |             device=self._lambda_v_table.device if device is None else device,
122 |         )
123 |         self._lambda_v_table.copy_(lambda_table.to(self._lambda_v_table))
124 |         return self._lambda_v_table
```

In [6]:
tdm = TrivialisedDiffusion(n_lambdas=128, lambda_num_batches=LAMBDA_NUM_BATCHES).to(device)
print(tdm)


TrivialisedDiffusion()


## 3. TDM Forward Diffusion

This is the exact source for the current forward path, followed by a real execution on the batch above.


In [7]:
show_source('src/kldm/diffusionModels/trivialized_diffusion.py', 129, 202, 'TDM forward_sample(...)')


### TDM forward_sample(...)

`src/kldm/diffusionModels/trivialized_diffusion.py:129-202`
```python
129 |     def forward_sample(
130 |         self,
131 |         t: torch.Tensor,
132 |         f0: torch.Tensor,
133 |         index: torch.Tensor,
134 |         v0: torch.Tensor | None = None,
135 |         epsilon_v: torch.Tensor | None = None,
136 |         epsilon_r: torch.Tensor | None = None,
137 |     ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
138 | 
139 | 
140 |         #Now we do T = [0,2] time scaling in order for TDM to converge.
141 |         t = self.time_scaling_T * t
142 | 
143 |         """
144 |         The transition kernel is defined as follow:
145 |             p_t|0 (ft, vt | f0, v0) = WN(r | wrapped_gaussian_mu_r_t, wrapped_gaussian_sigma_r_t) * Nv(vt | mu_v_t, sigma_v)
146 | 
147 |             transition kernel =          sample r_t              *         sample v_t
148 | 
149 |             We sample v_t, use it to move f_0 on manifold, to samlpe f_t
150 |         """
151 | 
152 |         #######################
153 |         ###    SAMPLE v_t   ###
154 |         #######################
155 | 
156 |         #Vi sætter v0 = 0, [Design choice] at time t = 0
157 |         if v0 is None:
158 |             v0 = torch.zeros_like(f0)                               #Design choice: Initial zero velocities
159 |         # Keep all state variables in native unit-period coordinates.
160 |         # Introducing angular 2π scaling here changes the optimization problem.
161 |         f0 = self.wrap_displacements(f0)
162 | 
163 |         #TODO: Scatter center mean free, også det de gør i KLDM
164 | 
165 |         #Sample normal noise for velocity                       # Nv is a normal distribution such that ∑vi = 0
166 |         if epsilon_v is None:
167 |             epsilon_v = torch.randn_like(v0)
168 |         epsilon_v = scatter_center(epsilon_v, index=index) #Zero mean
169 | 
170 |         gaussian_velocity_mean_coeff_t = self._match_dims(self.gaussian_velocity_mean_coeff(t), v0)
171 |         gaussian_velocity_sigma_t = self._match_dims(self.gaussian_velocity_sigma(t), v0)
172 | 
173 |         #Sample v_t, given initial velocity.
174 |         v_t = gaussian_velocity_mean_coeff_t * v0 + gaussian_velocity_sigma_t * epsilon_v
175 | 
176 |         ######################################
177 |         ###    Calculate displacement ft   ###
178 |         ######################################
179 |         #Now we calculate f_t = f_0 * expm(r_t), where r_t follows a wrapped Gaussian.
180 |         wrapped_gaussian_mu_r_t_pre_wrap = self.wrapped_gaussian_mu_r_t_pre_wrap(t, v_t, v0)
181 | 
182 |         wrapped_gaussian_sigma_r_t = self._match_dims(self.wrapped_gaussian_sigma_r_t(t), f0)
183 | 
184 |         #Sample normal noise on epsilon
185 |         if epsilon_r is None:
186 |             epsilon_r = torch.randn_like(f0)                        # Nr is a normal distribution such that ∑vi = 0
187 |         epsilon_r = scatter_center(epsilon_r, index=index)
188 | 
189 | 
190 |         # Native fractional pipeline: wrap in period 1 directly.
191 |         # Hidden angular scaling here would change both target norms and reverse dynamics.
192 |         r_t = self.wrap_displacements(wrapped_gaussian_mu_r_t_pre_wrap + wrapped_gaussian_sigma_r_t * epsilon_r)
193 | 
194 |         #Now we calculate displacement, and while we stay on the manifold.
195 |         f_t = self.wrap_displacements(f0 + r_t)
196 | 
197 |         #Center again
198 |         #f_t = scatter_center(f_t, index=index) DEACTIVED WHILE FRANCOIS ANSWERS MAIL.
199 |         #Then we wrap to ensure we stay within [0,1]
200 |         #f_t = self.wrap_positions(f_t)
201 | 
202 |         return f_t, v_t, epsilon_v, epsilon_r, r_t
```

In [8]:
batch_device = batch.to(device)
index = batch_device.batch
num_graphs = int(batch_device.num_graphs)

t_graph = torch.rand((num_graphs, 1), device=device)
t_node = t_graph[index].squeeze(-1)

f_t, v_t, epsilon_v, epsilon_r, r_t = tdm.forward_sample(
    t=t_node,
    f0=batch_device.pos,
    index=index,
)

summarize_tensor('t_graph', t_graph)
summarize_tensor('f_t', f_t)
summarize_tensor('v_t', v_t)
summarize_tensor('epsilon_v', epsilon_v)
summarize_tensor('epsilon_r', epsilon_r)
summarize_tensor('r_t', r_t)


t_graph: shape=(2, 1) dtype=torch.float32 mean=0.629331 std=0.407436 min=0.341230 max=0.917432
f_t: shape=(25, 3) dtype=torch.float32 mean=0.026667 std=0.273679 min=-0.475581 max=0.485628
v_t: shape=(25, 3) dtype=torch.float32 mean=-0.000000 std=0.904309 min=-2.441095 max=2.063535
epsilon_v: shape=(25, 3) dtype=torch.float32 mean=-0.000000 std=0.980735 min=-2.732084 max=2.391392
epsilon_r: shape=(25, 3) dtype=torch.float32 mean=0.000000 std=0.870018 min=-1.476971 max=2.572054
r_t: shape=(25, 3) dtype=torch.float32 mean=0.000000 std=0.282999 min=-0.498832 max=0.490396


## 4. TDM Training Target, Full Score Reconstruction, And Reverse Step

This is the wrapped-normal target used for training, the full score reconstruction used for sampling, and the reverse exponential step.


In [9]:
show_source('src/kldm/diffusionModels/trivialized_diffusion.py', 204, 305, 'TDM score_target(...) + construct_velocity_score(...) + reverse_exp_step(...)')


### TDM score_target(...) + construct_velocity_score(...) + reverse_exp_step(...)

`src/kldm/diffusionModels/trivialized_diffusion.py:204-305`
```python
204 |     def score_target(
205 |         self,
206 |         t: torch.Tensor,
207 |         # epsilon_v: torch.Tensor, not needed due to our initial velocity assumption
208 |         r_t: torch.Tensor,
209 |         v_t: torch.Tensor,
210 |         index: torch.Tensor,
211 |         v0: torch.Tensor | None = None,
212 |     ) -> torch.Tensor:
213 |         """Return the simplified TDM target predicted by the network.
214 | 
215 |         In the simplified parameterization we train on the stripped-down
216 |         wrapped-normal term only. The prefactor is reinserted later in
217 |         `construct_velocity_score(...)` when reconstructing Eq. (19).
218 |         """
219 | 
220 |         #We do time scaling.
221 |         t = self.time_scaling_T * t
222 | 
223 |         #Design choice, makes the target quite simple to calculate.
224 |         v0 = torch.zeros_like(v_t) if v0 is None else v0
225 | 
226 |         #Now we find target of the wrapped normal fractional distribution
227 |         # Keep target construction in exactly the same unit-period chart as training.
228 |         wrapped_gaussian_mu_r_t_pre_wrap = self.wrapped_gaussian_mu_r_t_pre_wrap(t, v_t, v0)
229 | 
230 |         # Keep the wrapped mean in the centered unit chart so the finite image truncation in
231 |         # d_log_wrapped_normal(...) stays numerically stable. Verify this TODO
232 |         wrapped_gaussian_mu_r_t = self.wrap_displacements(wrapped_gaussian_mu_r_t_pre_wrap)
233 | 
234 |         wrapped_gaussian_sigma_r_t = self._match_dims(self.wrapped_gaussian_sigma_r_t(t), r_t)
235 | 
236 | 
237 | 
238 |         wrapped_gaussian_target = d_log_wrapped_normal(
239 |             r=r_t,
240 |             mu=wrapped_gaussian_mu_r_t,
241 |             sigma=wrapped_gaussian_sigma_r_t,
242 |         )
243 | 
244 | 
245 | 
246 |         #Center the target
247 |         wrapped_gaussian_target = scatter_center(wrapped_gaussian_target, index=index)
248 | 
249 |         #target = gaussian_velocity_target + wrapped_gaussian_target
250 |         return wrapped_gaussian_target #Simplified version, might explain later.
251 | 
252 |         #return target_s
253 | 
254 |     def construct_velocity_score(
255 |         self,
256 |         t: torch.Tensor,
257 |         v_t: torch.Tensor,
258 |         pred_v: torch.Tensor,
259 |     ) -> torch.Tensor:
260 |         """Construct the full KLDM velocity score from the simplified prediction."""
261 |         t_internal = self.time_scaling_T * t
262 |         prefactor = self._match_dims(
263 |             (1.0 - torch.exp(-t_internal)) / (1.0 + torch.exp(-t_internal)),
264 |             pred_v,
265 |         )
266 |         gaussian_velocity_sigma_sq = self._match_dims(
267 |             self.gaussian_velocity_sigma(t_internal) ** 2,
268 |             pred_v,
269 |         ).clamp_min(self.eps)
270 | 
271 |         # Native period-1 reconstruction: no hidden chart conversion here.
272 |         return prefactor * pred_v - v_t / gaussian_velocity_sigma_sq
273 | 
274 |     def reverse_exp_step(
275 |         self,
276 |         f_t: torch.Tensor,
277 |         v_t: torch.Tensor,
278 |         score_v: torch.Tensor,
279 |         index: torch.Tensor,
280 |         dt: float,
281 |     ) -> tuple[torch.Tensor, torch.Tensor]:
282 |         """One exponential-integrator reverse step for the TDM process."""
283 |         f_t = self.wrap_displacements(f_t)
284 | 
285 |         dt_t = torch.as_tensor(
286 |             self.time_scaling_T * dt,
287 |             device=v_t.device,
288 |             dtype=v_t.dtype,
289 |         )
290 |         # Reverse noise must stay in the same unit-period scaling as v_t and score_v.
291 |         noise_v = scatter_center(torch.randn_like(v_t), index=index)
292 | 
293 |         exp_dt = torch.exp(dt_t)
294 |         exp_2dt_minus_1 = torch.exp(2.0 * dt_t) - 1.0
295 |         expm1_dt = torch.expm1(dt_t)
296 |         expm1_2dt = torch.expm1(2.0 * dt_t)
297 |         noise_scale = torch.sqrt(expm1_2dt.clamp_min(self.eps))
298 | 
299 |         # Keep reverse drift native too; mixing angular-sized score/noise here is an easy way
300 |         # to silently change the sampler.
301 |         v_prev = exp_dt * v_t + 2.0 * expm1_dt * score_v + noise_scale * noise_v
302 | 
303 |         f_prev = self.wrap_displacements(f_t - dt_t * v_prev)
304 | 
305 |         return f_prev, v_prev
```

In [10]:
target_v = tdm.score_target(
    t=t_node,
    r_t=r_t,
    v_t=v_t,
    index=index,
)
score_v = tdm.construct_velocity_score(
    t=t_node,
    v_t=v_t,
    pred_v=target_v,
)
f_prev, v_prev = tdm.reverse_exp_step(
    f_t=f_t,
    v_t=v_t,
    score_v=score_v,
    index=index,
    dt=1.0 / SAMPLING_STEPS,
)

summarize_tensor('target_v', target_v)
summarize_tensor('score_v', score_v)
summarize_tensor('f_prev', f_prev)
summarize_tensor('v_prev', v_prev)


target_v: shape=(25, 3) dtype=torch.float32 mean=-0.000000 std=2.343334 min=-5.810373 max=7.235379
score_v: shape=(25, 3) dtype=torch.float32 mean=0.000000 std=1.327833 min=-3.856736 max=4.014360
f_prev: shape=(25, 3) dtype=torch.float32 mean=-0.066667 std=0.248709 min=-0.486438 max=0.439918
v_prev: shape=(25, 3) dtype=torch.float32 mean=-0.000000 std=1.091273 min=-3.119549 max=2.001503


## 5. Lambda(t) Precomputation

This shows the exact loader-based Monte Carlo lambda code, then runs a small real precompute on the training loader.


In [11]:
show_source('src/kldm/diffusionModels/lambda_t.py', 1, 125, 'lambda_t.py')


### lambda_t.py

`src/kldm/diffusionModels/lambda_t.py:1-125`
```python
  1 | from __future__ import annotations
  2 | 
  3 | import torch
  4 | import torch.nn.functional as F
  5 | 
  6 | 
  7 | def interpolate_lambda_table(
  8 |     lambda_table: torch.Tensor,
  9 |     t01: torch.Tensor,
 10 | ) -> torch.Tensor:
 11 |     """
 12 |     Linearly interpolate a precomputed λ(t) table on normalized time t01 in [0, 1].
 13 |     """
 14 |     num_bins = lambda_table.shape[0]
 15 |     scaled = torch.clamp(t01, 0.0, 1.0) * (num_bins - 1)
 16 |     idx_lo = torch.floor(scaled).long()
 17 |     idx_hi = torch.clamp(idx_lo + 1, max=num_bins - 1)
 18 |     frac = (scaled - idx_lo.to(scaled.dtype)).to(lambda_table.dtype)
 19 | 
 20 |     lambda_lo = lambda_table[idx_lo]
 21 |     lambda_hi = lambda_table[idx_hi]
 22 |     return lambda_lo + (lambda_hi - lambda_lo) * frac
 23 | 
 24 | 
 25 | @torch.no_grad()
 26 | def precompute_lambda_time_grid_from_loader(
 27 |     diffusion,
 28 |     loader,
 29 |     t01_grid: torch.Tensor,
 30 |     num_batches: int = 32,
 31 |     clamp_min: float = 0.2,
 32 |     clamp_max: float = 5.0,
 33 |     smooth: bool = True,
 34 |     device: torch.device | None = None,
 35 | ) -> torch.Tensor:
 36 |     """
 37 |     Monte Carlo estimate of λ(t) for the exact wrapped-normal target used in training:
 38 | 
 39 |         λ(t) = 1 / E[ mean( target(t)^2 ) + eps ]
 40 | 
 41 |     where `target(t)` is the return value of `diffusion.score_target(...)`.
 42 | 
 43 |     This is intentionally based on the stripped wrapped-normal training target,
 44 |     because that is the quantity the network is regressed against.
 45 |     """
 46 |     if num_batches <= 0:
 47 |         raise ValueError("num_batches must be positive for lambda precomputation.")
 48 | 
 49 |     table_device = t01_grid.device if device is None else torch.device(device)
 50 |     num_bins = int(t01_grid.shape[0])
 51 | 
 52 |     sq_norm_sums = torch.zeros(num_bins, device=table_device, dtype=t01_grid.dtype)
 53 |     counts = torch.zeros(num_bins, device=table_device, dtype=t01_grid.dtype)
 54 | 
 55 |     for batch_idx, batch in enumerate(loader):
 56 |         if batch_idx >= num_batches:
 57 |             break
 58 | 
 59 |         batch = batch.to(table_device)
 60 |         index = batch.batch
 61 |         f0 = batch.pos.to(table_device)
 62 | 
 63 |         num_nodes = int(index.shape[0])
 64 |         num_graphs = int(batch.num_graphs)
 65 | 
 66 |         # One random time bin per graph, then broadcast it to that graph's nodes.
 67 |         graph_bin_idx = torch.randint(0, num_bins, (num_graphs,), device=table_device)
 68 |         node_bin_idx = graph_bin_idx[index]
 69 |         t_graph = t01_grid[graph_bin_idx]
 70 |         t_node = t_graph[index]
 71 | 
 72 |         _, v_t, _, _, r_t = diffusion.forward_sample(
 73 |             t=t_node,
 74 |             f0=f0,
 75 |             index=index,
 76 |         )
 77 | 
 78 |         target = diffusion.score_target(
 79 |             t=t_node,
 80 |             r_t=r_t,
 81 |             v_t=v_t,
 82 |             index=index,
 83 |         )
 84 | 
 85 |         # same reduction style as training loss
 86 |         node_sq = target.reshape(num_nodes, -1).pow(2).mean(dim=1)
 87 | 
 88 |         # Match the node-weighted averaging induced by the actual training loss.
 89 |         sq_norm_sums.scatter_add_(0, node_bin_idx, node_sq)
 90 |         counts.scatter_add_(0, node_bin_idx, torch.ones_like(node_sq))
 91 | 
 92 |     if not torch.any(counts > 0):
 93 |         raise RuntimeError("Train loader is empty; cannot precompute lambda(t).")
 94 | 
 95 |     expected_sq = sq_norm_sums / counts.clamp_min(1.0)
 96 | 
 97 |     # fill empty bins by nearest valid bin
 98 |     missing = counts <= 0
 99 |     if torch.any(missing):
100 |         valid_idx = torch.nonzero(counts > 0, as_tuple=False).squeeze(-1)
101 |         all_idx = torch.arange(num_bins, device=table_device)
102 |         nearest_valid = valid_idx[
103 |             torch.argmin(torch.abs(all_idx[:, None] - valid_idx[None, :]), dim=1)
104 |         ]
105 |         expected_sq = expected_sq.clone()
106 |         expected_sq[missing] = expected_sq[nearest_valid[missing]]
107 | 
108 |     # DSM-style inverse expected squared target norm
109 |     lambda_table = 1.0 / expected_sq.clamp_min(diffusion.eps)
110 | 
111 |     # smooth in log-space for stability
112 |     if smooth and num_bins >= 5:
113 |         log_lambda = torch.log(lambda_table.clamp_min(diffusion.eps))
114 |         kernel = torch.tensor([1.0, 2.0, 3.0, 2.0, 1.0], device=table_device, dtype=log_lambda.dtype)
115 |         kernel = kernel / kernel.sum()
116 |         x = F.pad(log_lambda[None, None, :], (2, 2), mode="replicate")
117 |         log_lambda = F.conv1d(x, kernel[None, None, :]).squeeze(0).squeeze(0)
118 |         lambda_table = torch.exp(log_lambda)
119 | 
120 |     # normalize around mean 1 for optimizer stability
121 |     lambda_table = lambda_table / lambda_table.mean().clamp_min(diffusion.eps)
122 | 
123 |     # mild clipping only
124 |     lambda_table = lambda_table.clamp(min=clamp_min, max=clamp_max)
125 |     return lambda_table
```

In [12]:
lambda_loader = task.dataloader(
    root=ROOT,
    split=SPLIT,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    download=DOWNLOAD,
)

lambda_table = tdm.precompute_lambda_v_table_from_loader(
    lambda_loader,
    device=device,
    num_batches=LAMBDA_NUM_BATCHES,
)

summarize_tensor('lambda_table', lambda_table)
print('lambda quantiles:', {
    'p25': float(torch.quantile(lambda_table, 0.25)),
    'p50': float(torch.quantile(lambda_table, 0.50)),
    'p75': float(torch.quantile(lambda_table, 0.75)),
})


lambda_table: shape=(128,) dtype=torch.float32 mean=1.137002 std=1.410906 min=0.200000 max=3.258193
lambda quantiles: {'p25': 0.20000000298023224, 'p50': 0.20000000298023224, 'p75': 3.258193254470825}


## 6. KLDM Algorithm 1, Weighted MSE, And Algorithm 2

This is the exact loss path the current model uses.


In [13]:
show_source('src/kldm/kldm.py', 72, 213, 'ModelKLDM: algorithm1_training_targets(...) + mse_loss_per_sample(...) + algorithm2_loss(...)')


### ModelKLDM: algorithm1_training_targets(...) + mse_loss_per_sample(...) + algorithm2_loss(...)

`src/kldm/kldm.py:72-213`
```python
 72 |     def algorithm1_training_targets(
 73 |         self,
 74 |         batch: Data | Batch,
 75 |         t: torch.Tensor,
 76 |     ) -> tuple[tuple[torch.Tensor, ...], tuple[torch.Tensor, ...]]:
 77 |         """
 78 |         Algorithm 1 in KLDM:
 79 |         sample noisy variables and score targets.
 80 |         """
 81 |         device = next(self.parameters()).device
 82 |         batch = batch.to(device)
 83 | 
 84 |         t_graph = t.to(device)
 85 |         if t_graph.ndim == 1:
 86 |             t_graph = t_graph[:, None]
 87 | 
 88 |         index = batch.batch
 89 |         t_node = t_graph[index].squeeze(-1)
 90 |         # Diffuse lattice, KLDM Alg. 1
 91 |         l_t, eps_l = self.diffusion_l.forward_sample(
 92 |             t=t_graph.squeeze(-1),
 93 |             x0=batch.l,
 94 |         )
 95 |         target_l = eps_l
 96 | 
 97 |         f_t, v_t, epsilon_v, epsilon_r, r_t = self.tdm.forward_sample(
 98 |             t=t_node,
 99 |             f0=batch.pos,
100 |             index=index,
101 |         )
102 | 
103 |         target_v = self.tdm.score_target(
104 |             t=t_node,
105 |             r_t=r_t,
106 |             v_t=v_t,
107 |             index=index,
108 |         )
109 | 
110 | 
111 |         return (v_t, f_t, l_t), (target_v, target_l)
112 | 
113 |     # ============================================================================
114 |     # Loss calculators for algorithm 2
115 |     # ============================================================================
116 | 
117 |     def mse_loss_per_sample(
118 |         self,
119 |         pred: torch.Tensor,
120 |         target: torch.Tensor,
121 |     ) -> torch.Tensor:
122 |         """
123 |         Plain MSE, averaged over feature dims.
124 |         """
125 |         loss = F.mse_loss(pred, target, reduction="none")
126 |         return loss.reshape(loss.shape[0], -1).mean(dim=1)
127 | 
128 |     # ============================================================================
129 |     # ALGORITHM 2
130 |     # ============================================================================
131 | 
132 |     def algorithm2_loss(
133 |         self,
134 |         batch: Data | Batch,
135 |         t: torch.Tensor,
136 |         lambda_v: float = 1.0,
137 |         lambda_l: float = 1.0,
138 |         debug: bool = False,
139 |     ) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
140 |         """
141 |         Algorithm 2 in KLDM:
142 |         network prediction + denoising score matching loss.
143 |         """
144 |         device = next(self.parameters()).device
145 |         batch = batch.to(device)
146 | 
147 |         t_graph = t.to(device)
148 |         if t_graph.ndim == 1:
149 |             t_graph = t_graph[:, None]
150 | 
151 |         index = batch.batch
152 |         # Algorithm 1
153 |         noisy, targets = self.algorithm1_training_targets(batch=batch, t=t_graph)
154 | 
155 |         v_t, f_t, l_t = noisy
156 |         target_v, target_l = targets
157 |         a_t = batch.h
158 | 
159 |         # Network prediction, KLDM Alg. 2
160 |         preds = self.score_network(
161 |             t=t_graph,
162 |             pos=f_t,
163 |             v=v_t,
164 |             h=a_t,
165 |             l=l_t,
166 |             node_index=index,
167 |             edge_node_index=batch.edge_node_index,
168 |         )
169 | 
170 |         ########HERE WE CALCULATE SIMPLIFIED SCORE
171 |         out_v = preds["v"]
172 |         out_l = preds["l"]
173 | 
174 |         # KLDM: plain squared error for lattice targets.
175 |         loss_l = self.mse_loss_per_sample(out_l, target_l).mean()
176 | 
177 |         # Precomputed λ(t) weighting on the simplified velocity target.
178 |         lambda_v_t = self.tdm.lambda_v(t_graph.squeeze(-1))[index]
179 |         raw_loss_v_per_sample = self.mse_loss_per_sample(out_v, target_v)
180 |         loss_v = (lambda_v_t * raw_loss_v_per_sample).mean()
181 |         raw_loss_v = raw_loss_v_per_sample.mean()
182 | 
183 |         total_loss = lambda_v * loss_v + lambda_l * loss_l
184 |         metrics = {
185 |             "loss": total_loss.detach(),
186 |             "loss_v": loss_v.detach(),
187 |             "loss_l": loss_l.detach(),
188 |         }
189 |         if debug:
190 |             t_node = t_graph[index].squeeze(-1)
191 |             score_v = self.tdm.construct_velocity_score(t=t_node, v_t=v_t, pred_v=out_v)
192 |             metrics.update(
193 |                 {
194 |                     "raw_loss_v": raw_loss_v.detach(),
195 |                     "target_v_abs_mean": target_v.abs().mean().detach(),
196 |                     "target_v_norm_mean": target_v.norm(dim=-1).mean().detach(),
197 |                     "pred_v_abs_mean": out_v.abs().mean().detach(),
198 |                     "pred_v_norm_mean": out_v.norm(dim=-1).mean().detach(),
199 |                     "lambda_v_mean": lambda_v_t.mean().detach(),
200 |                     "lambda_v_min": lambda_v_t.min().detach(),
201 |                     "lambda_v_max": lambda_v_t.max().detach(),
202 |                     "lambda_v_effective": (
203 |                         loss_v.detach() / raw_loss_v.detach().clamp_min(self.eps)
204 |                     ),
205 |                     "v_t_abs_mean": v_t.abs().mean().detach(),
206 |                     "f_t_abs_mean": f_t.abs().mean().detach(),
207 |                     "r_t_abs_mean": self.tdm.wrap_displacements(f_t - self.tdm.wrap_displacements(batch.pos)).abs().mean().detach(),
208 |                     "score_v_abs_mean": score_v.abs().mean().detach(),
209 |                     "pred_l_abs_mean": out_l.abs().mean().detach(),
210 |                     "target_l_abs_mean": target_l.abs().mean().detach(),
211 |                 }
212 |             )
213 |         return total_loss, metrics
```

In [14]:
model = ModelKLDM(device=device).to(device)
model.tdm.precompute_lambda_v_table_from_loader(
    lambda_loader,
    device=device,
    num_batches=LAMBDA_NUM_BATCHES,
)

loss, metrics = model.algorithm2_loss(
    batch=batch_device,
    t=t_graph,
    lambda_v=1.0,
    lambda_l=1.0,
    debug=True,
)

print(f'loss={float(loss):.6f}')
for key in sorted(metrics):
    value = metrics[key]
    if torch.is_tensor(value):
        print(f'{key}: {float(value):.6f}')
    else:
        print(f'{key}: {value}')


loss=2.426094
f_t_abs_mean: 0.235495
lambda_v_effective: 0.202338
lambda_v_max: 1.903485
lambda_v_mean: 1.017673
lambda_v_min: 0.200000
loss: 2.426094
loss_l: 1.027195
loss_v: 1.398899
pred_l_abs_mean: 0.292861
pred_v_abs_mean: 0.105104
pred_v_norm_mean: 0.194972
r_t_abs_mean: 0.230876
raw_loss_v: 6.913666
score_v_abs_mean: 0.833123
target_l_abs_mean: 0.712689
target_v_abs_mean: 1.761993
target_v_norm_mean: 3.174828
v_t_abs_mean: 0.703780
